In [ ]:
import os
import xml.etree.ElementTree as ET
import shutil
from sklearn.model_selection import train_test_split

# --- CONFIGURATION DES CHEMINS ---
# Assurez-vous que ces dossiers existent dans votre projet après avoir dézippé le dataset
input_images_dir = "dataset/images"
input_annotations_dir = "dataset/annotations"
output_root = "data"

# Classes définies dans le dataset Face Mask Detection
classes = ["with_mask", "without_mask", "mask_weared_incorrect"]

In [ ]:
def convert_coordinates(size, box):
    """Calcule les coordonnées normalisées pour YOLO (x_center, y_center, width, height)"""
    dw = 1. / size[0]
    dh = 1. / size[1]
    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0
    w = box[1] - box[0]
    h = box[3] - box[2]
    return (x * dw, y * dh, w * dw, h * dh)

In [ ]:
def convert_xml_to_txt(xml_file, output_txt_path):
    """Lit un fichier XML et écrit les labels au format YOLO dans un fichier TXT"""
    if not os.path.exists(xml_file):
        return False
    
    tree = ET.parse(xml_file)
    root = tree.getroot()
    size = root.find('size')
    w = int(size.find('width').text)
    h = int(size.find('height').text)

    with open(output_txt_path, 'w') as out_file:
        for obj in root.iter('object'):
            cls_name = obj.find('name').text
            if cls_name not in classes:
                continue
            cls_id = classes.index(cls_name)
            
            xmlbox = obj.find('bndbox')
            b = (float(xmlbox.find('xmin').text), 
                 float(xmlbox.find('xmax').text), 
                 float(xmlbox.find('ymin').text), 
                 float(xmlbox.find('ymax').text))
            
            bb = convert_coordinates((w, h), b)
            out_file.write(f"{cls_id} {' '.join(map(str, bb))}\n")
    return True

In [ ]:
# --- EXÉCUTION DU SCRIPT ---

# 1. Vérification des dossiers sources
if not os.path.exists(input_images_dir) or not os.path.exists(input_annotations_dir):
    print("Erreur : Les dossiers 'images' ou 'annotations' sont introuvables dans le dossier 'dataset'.")
else:
    # 2. Lister les noms de fichiers (sans extension)
    all_files = [f.split('.')[0] for f in os.listdir(input_images_dir) if f.endswith('.png')]
    
    # 3. Séparation Train (80%) / Val (20%)
    train_names, val_names = train_test_split(all_files, test_size=0.2, random_state=42)
    print(f"Total images : {len(all_files)}")
    print(f"Images pour l'entraînement : {len(train_names)}")
    print(f"Images pour la validation : {len(val_names)}")

    # 4. Création de la structure de dossiers YOLO
    for split in ['train', 'val']:
        os.makedirs(os.path.join(output_root, split, 'images'), exist_ok=True)
        os.makedirs(os.path.join(output_root, split, 'labels'), exist_ok=True)

    # 5. Fonction de traitement pour copier et convertir
    def process_split(names, split_type):
        for name in names:
            # Chemins sources
            src_img = os.path.join(input_images_dir, f"{name}.png")
            src_xml = os.path.join(input_annotations_dir, f"{name}.xml")
            
            # Chemins destinations
            dst_img = os.path.join(output_root, split_type, 'images', f"{name}.png")
            dst_label = os.path.join(output_root, split_type, 'labels', f"{name}.txt")
            
            # Copie de l'image
            shutil.copy(src_img, dst_img)
            
            # Conversion de l'annotation
            convert_xml_to_txt(src_xml, dst_label)

    print("Traitement des fichiers en cours...")
    process_split(train_names, 'train')
    process_split(val_names, 'val')
    print("Succès : Le dataset est prêt dans le dossier 'data/' !")

Total images : 853
Images pour l'entraînement : 682
Images pour la validation : 171
Traitement des fichiers en cours...
Succès : Le dataset est prêt dans le dossier 'data/' !
